## Script for Comparing Incorrectly Labeled Epochs

In [ ]:
import mne 
import os
import numpy as np 
import pandas as pd
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import pickle
from sklearn.model_selection import train_test_split, KFold
import matplotlib
matplotlib.use('QtAgg') 

In [ ]:
# importing model 
with open("M2_all.pkl", "rb") as f:
    model = pickle.load(f)

In [ ]:
# importing features dataframe 
features_all = pd.read_pickle("training_features_19032026.pkl")

In [ ]:
X_zygo = features_all[["Zygo"]] 
X_corr = features_all[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

y_zygo = features_all["Num_Contractions_Zygo"].astype(int).to_numpy()
y_corr = features_all["Num_Contractions_Corr"].astype(int).to_numpy()

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate((y_zygo,y_corr),axis=0)       


indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

In [ ]:
n1 = len(X_zygo)
labels = np.where(idx_test < n1, "Zygo", "Corr")

idx_test_1 = idx_test[idx_test < n1]
idx_test_2 = idx_test[idx_test >= n1] - n1

In [ ]:
X_test_zygo = X[idx_test_1]
X_test_corr = X[idx_test_2]
X_test = X[idx_test]

# get testing results 
y_pred_test_zygo = model.predict(X_test_zygo)   
y_pred_test_corr = model.predict(X_test_corr)   
y_pred_test  = model.predict(X_test)

y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)   
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)   
y_pred_test = np.argmax(y_pred_test,axis=1)

In [ ]:
subjects = features_all["Subject"] 
subject_subset = np.concatenate([subjects[idx_test_1],subjects[idx_test_2]])

epochs = features_all["Triggers_Order_Nap"] 
epochs_subset = np.concatenate([epochs[idx_test_1],epochs[idx_test_2]])

naps = features_all["Nap Number"] 
naps_subset = np.concatenate([naps[idx_test_1],naps[idx_test_2]])

In [ ]:
# get mismatched epoch indices where the tested indices dont equal predicted 
epoch_idx = np.where(y[idx_test] != y_pred_test)

mismatched_subj = subject_subset[epoch_idx]
mismatched_epochs = epochs_subset[epoch_idx]
mismatched_naps = naps_subset[epoch_idx]


Pre-Processing

In [31]:
# making dataframe for epoch rescoring 
rescore_epochs = pd.DataFrame({
    "Subject": mismatched_subj,
    "Nap Number": mismatched_naps,
    "Triggers_Order_Nap":mismatched_epochs,
    "Label": labels[epoch_idx],
    "Prediction": y_pred_test[epoch_idx]
}, index=epoch_idx[0])

df_wide = (
    rescore_epochs
    .pivot_table(
        index=["Subject", "Nap Number", "Triggers_Order_Nap"],
        columns="Label",
        values="Prediction",
        aggfunc="first"
    )
    .rename(columns={
        "Zygo": "Prediction_Zygo",
        "Corr": "Prediction_Corr"
    })
    .reset_index()
)

In [36]:
print(df_wide.head())

Label Subject  Nap Number  Triggers_Order_Nap  Prediction_Corr  \
0      NL03JV           1                  32              0.0   
1      NL03JV           1                  46              NaN   
2      NL03JV           2                   1              3.0   
3      NL03JV           3                  43              3.0   
4      NL03JV           4                  30              3.0   

Label  Prediction_Zygo  
0                  NaN  
1                  3.0  
2                  NaN  
3                  NaN  
4                  NaN  


In [49]:
rows = []
seen = set()

for _, row in df_wide.iterrows():
    key = (row["Subject"], row["Nap Number"], row["Triggers_Order_Nap"])

    # skip duplicates
    if key in seen:
        continue
    seen.add(key)

    # find matching row(s) in the other dataframe
    match = features_all[
        (features_all["Subject"] == row["Subject"]) &
        (features_all["Nap Number"] == row["Nap Number"]) &
        (features_all["Triggers_Order_Nap"] == row["Triggers_Order_Nap"])
    ]

    # if there is a match, take the first one
    if not match.empty:
        match_row = match.iloc[0]

        # --- HANDLE NaNs HERE ---
        match_zygo = match_row["Zygo"]
        match_corr = match_row["Corr"]

        pred_zygo = row["Prediction_Zygo"]
        pred_corr = row["Prediction_Corr"]

        if pd.isna(pred_zygo):
            pred_zygo = np.argmax(model.predict(np.array(match_zygo.tolist()).reshape(1, 2251,1), verbose=0))

        if pd.isna(pred_corr):
            pred_corr = np.argmax(model.predict(np.array(match_corr.tolist()).reshape(1, 2251,1), verbose=0))


        rows.append({
            "Subject": row["Subject"],
            "Nap Number": row["Nap Number"],
            "Triggers_Order_Nap": row["Triggers_Order_Nap"],
            "Prediction_Zygo": pred_zygo,
            "Prediction_Corr": pred_corr,

            # examples of fields from df_other
            "Num_Contractions_Zygo": match_row["Num_Contractions_Zygo"],
            "Num_Contractions_Corr": match_row["Num_Contractions_Corr"],
            "Zygo": match_row["Zygo"],
            "Corr": match_row["Corr"],
        })

    else:
        # if no match exists, still keep the row
        rows.append({
            "Subject": row["Subject"],
            "Nap Number": row["Nap Number"],
            "Triggers_Order_Nap": row["Triggers_Order_Nap"],
            "Label": row["Label"],
            "Prediction_Zygo": pred_zygo,
            "Prediction_Corr": pred_corr,

            "Num_Contractions_Zygo": pd.NA,
            "Num_Contractions_Corr": pd.NA,
            "Zygo": pd.NA,
            "Corr": pd.NA
        })

new_df = pd.DataFrame(rows)    


In [51]:
new_df.head(50)

,Subject,Nap Number,Triggers_Order_Nap,Prediction_Zygo,Prediction_Corr,Num_Contractions_Zygo,Num_Contractions_Corr,Zygo,Corr
0,NL03JV,1,32,0.0,0.0,0,3,"[-7.852191152235275, 22.36489473697002, 38.886...","[0.5594635643762711, 1.9752581913113216, 3.063..."
1,NL03JV,1,46,3.0,0.0,0,0,"[0.8492664154353681, -4.43875545007432, 0.2553...","[5.068297668329948, 0.20087344125415907, -2.04..."
2,NL03JV,2,1,3.0,3.0,3,0,"[0.17996856344134926, 0.4789064147138773, -2.4...","[-1.1927338791176654, -5.049525399424987, -2.6..."
3,NL03JV,3,43,0.0,3.0,0,0,"[3.1105101337936567, 2.1467638629543746, -1.80...","[2.8319603600311805, 4.698823859226004, 6.1783..."
4,NL03JV,4,30,0.0,3.0,0,3,"[0.6582743944272096, -0.5405055157722352, 1.28...","[-1.6499395053667607, -1.8761447433472902, 0.5..."
5,NL03JV,5,22,0.0,0.0,3,0,"[-1.033818309589727, 1.7559225705198656, 0.523...","[-3.660634886869012, -11.005626278819609, -4.8..."
6,NL03JV,5,26,0.0,0.0,0,3,"[-20.76444069460169, -302.64640790748433, -253...","[14.579065501942594, 3.125195734931459, -1.563..."
7,NL04NF,1,32,0.0,0.0,0,3,"[5.0746382948915505, -1.6452872537999674, -1.2...","[-1.821233739452798, -3.1199131288115485, -0.8..."
8,NL04NF,1,53,0.0,0.0,0,3,"[6.921233940636412, -3.2273095323349605, 1.348...","[2.1580087907348795, -0.09169418635615645, -0...."
9,NL04NF,2,54,0.0,0.0,0,0,"[0.21052166695559427, -2.860457607843262, -1.3...","[2.2911904308566164, 1.964510291285604, 5.4070..."


Scoring for Mismatched Epochs

In [ ]:
#GUI

### For now plotting ECG instead of corru
frq=250 # hardcode 250 
last_epoch_length = 5
current_index=0

def plot_figure(t):
    global df_triggers
    fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    if df_triggers.loc[t, 'Contraction_number'] != None and df_triggers.loc[t, 'Response_end_sample'] != None and df_triggers.loc[t, 'Response_start_sample'] != None:
        fdct = {'color': 'r'}
        title = fig.suptitle(f"Epoch {t + 1} - SCORED : {df_triggers.loc[t, 'Contraction_number']} CONTRACTIONS; START = {df_triggers.loc[t, 'Response_start_sample']}; END = {df_triggers.loc[t, 'Response_end_sample']}")
        title.set(**fdct)
    elif df_triggers.loc[t, 'Contraction_number'] ==0 and df_triggers.loc[t, 'Response_end_sample'] == None and df_triggers.loc[t, 'Response_start_sample'] == None:
        fdct = {'color': 'r'}
        title = fig.suptitle(f"Epoch {t + 1} - SCORED : {df_triggers.loc[t, 'Contraction_number']} CONTRACTIONS")
        title.set(**fdct)
    else:
        title = fig.suptitle(f"Epoch {t + 1} - UNFINISHED SCORING : {df_triggers.loc[t, 'Contraction_number']} CONTRACTIONS; START = {df_triggers.loc[t, 'Response_start_sample']}; END = {df_triggers.loc[t, 'Response_end_sample']}")

    # Plot Corru (ECG for now)
    if t==239 or df_triggers['Stim_time_sample'][t+1]-df_triggers['Stim_time_sample'][t]>10*frq:
        ax[0].plot(range(int(df_triggers['Stim_time_sample'][t]-20),int(df_triggers['Stim_time_sample'][t]+last_epoch_length*frq)), raw_wEEG_wZygo.get_data(picks=['ECG'],start=int(df_triggers['Stim_time_sample'][t]-20),stop=int(df_triggers['Stim_time_sample'][t]+last_epoch_length*frq))[0], label="Chin", color="blue")
    else:    
        ax[0].plot(range(int(df_triggers['Stim_time_sample'][t]-20),int(df_triggers['Stim_time_sample'][t+1]-1)), raw_wEEG_wZygo.get_data(picks=['ECG'],start=int(df_triggers['Stim_time_sample'][t]-20),stop=int(df_triggers['Stim_time_sample'][t+1]-1))[0], label="Chin", color="blue")
    #ax[0].set_ylim(-0.0002, 0.0002)
    ax[0].set_ylabel("ECG")

    # Plot Zygo
    if t==239 or df_triggers['Stim_time_sample'][t+1]-df_triggers['Stim_time_sample'][t]>10*frq:
        ax[1].plot(range(int(df_triggers['Stim_time_sample'][t]-20),int(df_triggers['Stim_time_sample'][t]+last_epoch_length*frq)), raw_wEEG_wZygo.get_data(picks=['Zygo'],start=int(df_triggers['Stim_time_sample'][t]-20),stop=int(df_triggers['Stim_time_sample'][t]+last_epoch_length*frq))[0], label="Zygo", color="black")
    else:   
        ax[1].plot(range(int(df_triggers['Stim_time_sample'][t]-20),int(df_triggers['Stim_time_sample'][t+1]-1)), raw_wEEG_wZygo.get_data(picks=['Zygo'],start=int(df_triggers['Stim_time_sample'][t]-20),stop=int(df_triggers['Stim_time_sample'][t+1]-1))[0], label="Zygo", color="black")
    ax[1].set_ylim(-0.0002, 0.0002)
    ax[1].set_ylabel("Zygo")

    # Plot the trigger channel
    if t==239 or df_triggers['Stim_time_sample'][t+1]-df_triggers['Stim_time_sample'][t]>10*frq:
        ax[2].plot(range(int(df_triggers['Stim_time_sample'][t]-20),int(df_triggers['Stim_time_sample'][t]+last_epoch_length*frq)), raw_wEEG_wZygo.get_data(picks=['Trigger'],start=int(df_triggers['Stim_time_sample'][t]-20),stop=int(df_triggers['Stim_time_sample'][t]+last_epoch_length*frq))[0], label="Trigger Channel", color="orange")
    else:
        ax[2].plot(range(int(df_triggers['Stim_time_sample'][t]-20),int(df_triggers['Stim_time_sample'][t+1]-1)), raw_wEEG_wZygo.get_data(picks=['Trigger'],start=int(df_triggers['Stim_time_sample'][t]-20),stop=int(df_triggers['Stim_time_sample'][t+1]-1))[0], label="Trigger Channel", color="orange")
    
    ax[2].set_xlabel("Stim_time_sample")
    ax[2].set_ylabel("Trigger")
    ax[2].set_ylim(0, 40)

    # Connect mouse click and key press events
    fig.canvas.mpl_connect('key_press_event', on_key)
    fig.canvas.mpl_connect('button_press_event', on_click)

    plt.tight_layout()
    plt.show()

# Event handling function
def on_click(event):
    global current_index, fig, df_triggers
    if event.inaxes:  # Check if click occurred in any axes
        for i, a in enumerate(event.canvas.figure.axes):
            if event.inaxes == a:  # Check which subplot was clicked
                # print(f"Clicked in Subplot {i+1}")
                # print(f"Coordinates in data space: X = {event.xdata}, Y = {event.ydata}")
                if df_triggers['Muscle_type'][current_index]  == None:
                    df_triggers.loc[current_index, 'Response_start_sample' ] = int(event.xdata)
                    df_triggers.loc[current_index, 'RT_sec' ] = (df_triggers.loc[current_index, 'Response_start_sample' ] - df_triggers.loc[current_index, 'Stim_time_sample'])/frq
                    if i==0:
                        df_triggers.loc[current_index, 'Muscle_type'] = "Corru"
                    elif i==1:
                        df_triggers.loc[current_index, 'Muscle_type'] = "Zygo"           
                else:
                    df_triggers.loc[current_index, 'Response_end_sample' ] = int(event.xdata)

    if (df_triggers.loc[current_index, 'Contraction_number'] != None and df_triggers.loc[current_index, 'Response_end_sample'] != None and df_triggers.loc[current_index, 'Response_start_sample'] != None) or (df_triggers.loc[current_index, 'Contraction_number'] ==0):
        plt.close()  # Close current figure
        plot_figure(current_index)         
                    

# Keyboard press event handler
def on_key(event):
    global current_index, fig, df_triggers
    allowed_keys = {'0','1', '2', '3', '4', '5','5', '7', '8', '9'}
    # print(f"Key pressed: {event.key}")  # Print the key pressed
    key = event.key

    if key in allowed_keys:
        df_triggers.loc[current_index, 'Contraction_number'] = int(key) 
        if (df_triggers.loc[current_index, 'Response_end_sample'] != None and df_triggers.loc[current_index, 'Response_start_sample'] != None) or (df_triggers.loc[current_index, 'Contraction_number'] ==0):
            plt.close()  # Close current figure
            plot_figure(current_index) 
        
    elif event.key == 'right':  # Move to next figure
        current_index = (current_index + 1) % len(df_triggers)  # Loop to the start
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the next figure
    elif event.key == 'left':  # Move to previous figure
        current_index = (current_index - 1) % len(df_triggers) # Loop to the end
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'r':  # Move to previous figure
        df_triggers.loc[current_index, 'Response_start_sample'] = None
        df_triggers.loc[current_index, 'Response_end_sample'] = None 
        df_triggers.loc[current_index, 'Contraction_number'] = None
        df_triggers.loc[current_index, 'Muscle_type'] = None
        df_triggers.loc[current_index, 'RT_sec'] = None
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'q':  # Custom action for specific key
        print("Quitting the plot!")
        plt.close()  # Close the figure
        df_triggers.to_csv(path_metadata+'/subject_{}_{}_metadata.csv'.format(subject,task_type),index=False)
        

# Plot the first figure
plot_figure(current_index)